# QLoRA on T4 with Unsloth

Fine-tunes a small 4-bit (QLoRA) model with LoRA adapters on a 1,000-sample instruction dataset.

- LoRA will train (`print_trainable_parameters` → typically <1%).
- Adapter on disk will be a few MB, not GBs.

## 1. Install Unsloth

If this cell errors on a fresh Colab, swap to the nightly line in the comment.

In [1]:
%%capture
!pip install --upgrade --quiet unsloth
# If the stable wheel fails, uncomment the nightly:
# !pip uninstall -y unsloth && pip install --no-deps --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Load a 4-bit base model

Loading `unsloth/Llama-3.2-3B-Instruct` in 4-bit NF4. Base weights stay **frozen and quantized**; only the LoRA adapters (next cell) will train.

Smaller alternatives for faster iteration:
- `unsloth/Llama-3.2-1B-Instruct`
- `unsloth/gemma-2-2b-it`
- `unsloth/Qwen2.5-3B-Instruct`

In [2]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048   # Unsloth handles RoPE scaling
LOAD_IN_4BIT   = True   # the "Q" in QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,          # autodetect: T4 → fp16
    load_in_4bit   = LOAD_IN_4BIT,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


## 3. Attach LoRA adapters (the "low-rank" part)

With `r=16`, every targeted linear layer gets a rank-16 update - a tiny slice of the full weight matrix.

**Knobs worth playing with on re-runs:**
- `r`: 4 → 8 → 16 → 64 (capacity vs. parameter count)
- `target_modules`: just `["q_proj", "v_proj"]` for the minimal LoRA, vs. all 7 linear layers for maximal
- `lora_alpha`: usually `r` or `2*r`

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha     = 16,
    lora_dropout   = 0,        # 0 is the fast path in Unsloth
    bias           = "none",   # also the fast path
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = False,
    loftq_config   = None,
)

model.print_trainable_parameters()

Unsloth 2026.5.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## 4. Load a tiny dataset

`mlabonne/guanaco-llama2-1k` - 1,000 instruction examples, already formatted with Llama chat tags. Perfect smoke-test size.

Other small open datasets to drop in:
- `yahma/alpaca-cleaned` (52k - slice with `.select(range(1000))`)
- `databricks/databricks-dolly-15k`
- `HuggingFaceH4/no_robots` (10k, high quality)

In [4]:
from datasets import load_dataset

dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
print(dataset)
print("---")
print(dataset[0]["text"][:500])

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1000
})
---
<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal


## 5. Sample BEFORE fine-tuning

Generate once now so we can diff against the post-training output.

In [5]:
FastLanguageModel.for_inference(model)

PROMPT = "Explain the difference between LoRA and full fine-tuning in two sentences."

def generate(text: str) -> str:
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens = 128,
        temperature    = 0.7,
        do_sample      = True,
        use_cache      = True,
    )
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

before = generate(PROMPT)
print(before)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Explain the difference between LoRA and full fine-tuning in two sentences. LoRA (Low-Rank Approximation) is a technique used to fine-tune a pre-trained model by learning a lower-dimensional representation of the data, while full fine-tuning involves learning a new representation from scratch. This means that LoRA reduces the dimensionality of the feature space, whereas full fine-tuning increases the dimensionality of the feature space. 

Here is the code for LoRA fine-tuning:
```python
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import AutoModel

class LoRAFineTune(nn.Module):
    def __init__(self, model,


## 6. Train

60 steps over the 1k dataset is ≈10–15 min on a free T4. For a fuller run, set `num_train_epochs=1` (and drop `max_steps`).

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        max_steps                   = 60,        # raise for a fuller run
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | total VRAM: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Reserved before training: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

GPU: Tesla T4 | total VRAM: 14.6 GB
Reserved before training: 2.59 GB


In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,1.631504
10,1.612939
15,1.574296
20,1.453820
25,1.557466
30,1.614928
35,1.361096
40,1.417630
45,1.242101
50,1.422927


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


## 7. Sample AFTER fine-tuning

In [9]:
FastLanguageModel.for_inference(model)
after = generate(PROMPT)

print("=== BEFORE ===\n" + before)
print("\n=== AFTER ===\n"  + after)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


=== BEFORE ===
Explain the difference between LoRA and full fine-tuning in two sentences. LoRA (Low-Rank Approximation) is a technique used to fine-tune a pre-trained model by learning a lower-dimensional representation of the data, while full fine-tuning involves learning a new representation from scratch. This means that LoRA reduces the dimensionality of the feature space, whereas full fine-tuning increases the dimensionality of the feature space. 

Here is the code for LoRA fine-tuning:
```python
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import AutoModel

class LoRAFineTune(nn.Module):
    def __init__(self, model,

=== AFTER ===
Explain the difference between LoRA and full fine-tuning in two sentences. LoRA (Linearly Regressed Autoencoder) is a method of fine-tuning a pre-trained model by learning a linear combination of the model's weights, whereas full fine-tuning involves training the entire model from scratch using the pre-traine

## 8. Save the LoRA adapter

The adapter is just the trained delta - typically a few MB. Load it on top of the same base model later with `FastLanguageModel.from_pretrained("lora_model", ...)`.

In [10]:
import os

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

size_mb = sum(
    os.path.getsize(os.path.join("lora_model", f))
    for f in os.listdir("lora_model")
) / 1024**2
print(f"Adapter dir size: {size_mb:.1f} MB")

# Optional - push to Hugging Face Hub:
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your-username/qlora-llama32-3b-guanaco")
# tokenizer.push_to_hub("your-username/qlora-llama32-3b-guanaco")

Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


Adapter dir size: 109.3 MB


## 9. Things to try next

- Re-run from cell 3 with `r = 4`, then `r = 64`. Compare trainable-param count, adapter size, and quality of the AFTER sample.
- Restrict `target_modules` to just `["q_proj", "v_proj"]` - the original LoRA paper recipe - and see how much capacity is lost.
- Swap the dataset for `yahma/alpaca-cleaned` and slice to 500 examples: `load_dataset(...).select(range(500))`.
- Try a smaller base (`unsloth/Llama-3.2-1B-Instruct`) and observe how rank interacts with model size.
- Set `max_steps = -1` and `num_train_epochs = 1` for a fuller 1-epoch run over the dataset.